<a href="https://colab.research.google.com/github/vikashkumar-raj/AI-Candlestick-Prediction/blob/main/Gold_AI_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ==============================================================================
# 🥇 PHASE 1: DATA INGESTION & STRUCTURAL SCHEMA VALIDATION (SEMICOLON RESOLVED)
# Folder Mapping: src/data/loader.py
# ==============================================================================


print("==========================================================")
print("📥 PHASE 1: PARSING Semicolon DELIMITER & LOADING DATA")
print("==========================================================")

def execute_phase_1_with_drive():
    """
    Mounts drive, reads semicolon separated raw logs, unpacks features cleanly,
    validates the structural framework, and chronologically tracks entries.
    """
    # 1. Secure Drive connection checkpoint
    print("[1/4] Securing active Google Drive context...")
    drive.mount('/content/drive', force_remount=True)

    # 2. Hard-coded prioritized path matrix from previous step
    possible_paths = [
        '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv',
        '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/raw/XAU_1h.csv',
        '/content/drive/MyDrive/XAU_1h_data.csv',
        'XAU_1h_data.csv'
    ]

    target_path = None
    for path in possible_paths:
        if os.path.exists(path):
            target_path = path
            break

    if target_path is None:
        raise FileNotFoundError("🔴 Critical Error: File could not be located in the allocated directories.")

    print(f"[2/4] Parsing file source with Semicolon rules: '{target_path}'")

    # 3. Reading with semicolon specifier to parse distinct features cleanly
    raw_df = pd.read_csv(target_path, sep=';')

    # Correcting dynamic header text artifact if any mapping error happens
    raw_df.columns = [col.split(';')[0].strip() for col in raw_df.columns]

    # 4. Mandatory column checking baseline validation
    mandatory_schema = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    missing_columns = [col for col in mandatory_schema if col not in raw_df.columns]

    if missing_columns:
        raise KeyError(f"🔴 Schema Configuration Failed! Missing features: {missing_columns}")
    print("   ✓ Core OHLCV + Volume features split successfully.")

    # 5. Advanced formatting logic to handle timestamp variations (YYYY.MM.DD HH:MM)
    print("[3/4] Unpacking and standardizing time signatures...")
    raw_df['Date'] = raw_df['Date'].astype(str).str.replace('.', '-', regex=False)
    raw_df['Date'] = pd.to_datetime(raw_df['Date'], errors='raise')

    # Ensure numerical features are strictly typed to float32/int32 vectors
    for num_col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        raw_df[num_col] = pd.to_numeric(raw_df[num_col], errors='coerce')

    print("[4/4] Locking strict chronological historical array execution sequence...")
    processed_df = raw_df.sort_values('Date').reset_index(drop=True)

    # 6. Metadata profile print summary
    total_bars = len(processed_df)
    print("\n==========================================================")
    print("🎯 PHASE 1: REPOSITORY DATA INGESTION PROFILE REPORT")
    print("==========================================================")
    print(f"  • Resolved Path           : {target_path}")
    print(f"  • Total Time Bars Ingested: {total_bars}")
    print(f"  • Timeline Tracking From  : {processed_df['Date'].min()}")
    print(f"  • Timeline Tracking To    : {processed_df['Date'].max()}")
    print("==========================================================")
    print("✅ Phase 1 data blocks completely mapped and locked.")

    return processed_df

# Run Phase 1 execution
df_1h = execute_phase_1_with_drive()

📥 PHASE 1: PARSING Semicolon DELIMITER & LOADING DATA
[1/4] Securing active Google Drive context...
Mounted at /content/drive
[2/4] Parsing file source with Semicolon rules: '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv'
   ✓ Core OHLCV + Volume features split successfully.
[3/4] Unpacking and standardizing time signatures...
[4/4] Locking strict chronological historical array execution sequence...

🎯 PHASE 1: REPOSITORY DATA INGESTION PROFILE REPORT
  • Resolved Path           : /content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv
  • Total Time Bars Ingested: 125206
  • Timeline Tracking From  : 2004-06-11 07:00:00
  • Timeline Tracking To    : 2026-01-30 23:00:00
✅ Phase 1 data blocks completely mapped and locked.


In [4]:
# ==============================================================================
# 🥇 PHASE 2: EXPLORATORY DATA ANALYSIS & INTEGRITY VERIFICATION (1H TIMEFRAME)
# Folder Mapping: src/data/validator.py & src/data/cleaner.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🔍 PHASE 2: INITIALIZING CORE DATA INTEGRITY DIAGNOSTICS")
print("==========================================================")

def execute_phase_2_validation(df):
    """
    Runs multi-point integrity checks for numeric anomalies, zero valuation
    boundaries, time gaps, and duplicate rows across the loaded asset stream.
    """
    # Create an isolated local copy to prevent setting-with-copy warnings
    working_df = df.copy()

    print("[1/3] Scanning timeline dimensions for temporal anomalies...")
    # 1. Deduplicate timeline sequences based on sorted datetime index
    duplicate_count = working_df['Date'].duplicated().sum()
    if duplicate_count > 0:
        print(f"   ⚠️ Warning: Detected {duplicate_count} overlapping duplicate rows. Clearing duplicates...")
        working_df = working_df.drop_duplicates(subset=['Date']).reset_index(drop=True)
    else:
        print("   ✓ Integrity Pass: Timeline sequence has zero tracking overlaps.")

    print("[2/3] Auditing matrix cell arrays for NaN and Infinity values...")
    # 2. Extract numeric feature targets for statistical sanity testing
    numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']

    # Evaluate data leakage or broken metrics array cells
    nan_cells = working_df[numeric_columns].isna().sum().to_dict()
    inf_cells = np.isinf(working_df[numeric_columns]).sum().to_dict()

    print(f"   • Null/NaN Cell Map : {nan_cells}")
    print(f"   • Infinite Bound Map: {inf_cells}")

    # 3. Structural Boundary Verification: Ensure absolute prices are positive real numbers
    illegal_bounds = (working_df[['Open', 'High', 'Low', 'Close']] <= 0).sum().sum()
    if illegal_bounds > 0:
        raise ValueError("🔴 Data Invalidation: Negative or absolute zero market evaluation discovered in baseline matrix rows.")

    print("[3/3] Engineering underlying descriptive metrics report...")
    # 4. Generate high-precision asset baseline statistics
    ohlc_summary = working_df[numeric_columns].describe().T

    print("\n==========================================================")
    print("📊 PHASE 2: DATA INTEGRITY & PHYSICAL RUNTIME LOG")
    print("==========================================================")
    print(f"  • Post-Sanitization Records : {len(working_df)}")
    print(f"  • Asset Global Floor (Low)  : {working_df['Low'].min():.2f}")
    print(f"  • Asset Global Peak (High)  : {working_df['High'].max():.2f}")
    print(f"  • Mean Volume Distribution  : {working_df['Volume'].mean():.2f}")
    print("==========================================================")
    print("✅ Phase 2 diagnostic evaluations complete. Dataframe purity locked.")

    return working_df

# Trigger Phase 2 Execution using the df_1h vector array variable from Phase 1
df_1h = execute_phase_2_validation(df_1h)

🔍 PHASE 2: INITIALIZING CORE DATA INTEGRITY DIAGNOSTICS
[1/3] Scanning timeline dimensions for temporal anomalies...
   ✓ Integrity Pass: Timeline sequence has zero tracking overlaps.
[2/3] Auditing matrix cell arrays for NaN and Infinity values...
   • Null/NaN Cell Map : {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}
   • Infinite Bound Map: {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}
[3/3] Engineering underlying descriptive metrics report...

📊 PHASE 2: DATA INTEGRITY & PHYSICAL RUNTIME LOG
  • Post-Sanitization Records : 125206
  • Asset Global Floor (Low)  : 381.10
  • Asset Global Peak (High)  : 5597.60
  • Mean Volume Distribution  : 3900.01
✅ Phase 2 diagnostic evaluations complete. Dataframe purity locked.


In [5]:
# ==============================================================================
# 🥇 PHASE 3: CANDLESTICK DETECTION ENGINE (1H TIMEFRAME - SYNTAX FIXED)
# Folder Mapping: src/candlestick/detector.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🕯️ PHASE 3: INITIALIZING ARCHITECTURAL CANDLESTICK DETECTION")
print("==========================================================")

def execute_phase_3_candlestick_engine(df):
    """
    Vectorized structural mathematical engine that scans and maps 22 distinct
    candlestick formations precisely without shifting windows to preserve temporal purity.
    """
    working_df = df.copy()

    # --- HELPER BASE VALUATIONS (Vectorized Operations) ---
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()

    # Core body and shadow boundaries
    body = np.abs(C - O)
    candle_range = H - L
    # Prevent divide-by-zero on completely flat bars
    candle_range = np.where(candle_range == 0, 1e-5, candle_range)

    body_max = np.maximum(O, C)
    body_min = np.minimum(O, C)

    upper_shadow = H - body_max
    lower_shadow = body_min - L

    is_bullish = C > O
    is_bearish = C < O

    # Average body calculation for standard comparison metrics (rolling window context)
    avg_body = working_df['Close'].diff().abs().rolling(window=10, min_periods=1).mean().to_numpy()
    avg_body = np.nan_to_num(avg_body, nan=1e-5)

    # --------------------------------------------------------------------------
    # VECTORIZED STRUCTURAL PATTERN DETECTION LOGIC (22 PATTERNS)
    # --------------------------------------------------------------------------

    # 1. Doji
    working_df['Pattern_Doji'] = (body <= (candle_range * 0.1)).astype(int)

    # 2. Marubozu
    working_df['Pattern_Marubozu'] = ((body >= (candle_range * 0.9)) & (upper_shadow <= (candle_range * 0.05)) & (lower_shadow <= (candle_range * 0.05))).astype(int)

    # 3. Hammer
    working_df['Pattern_Hammer'] = ((lower_shadow >= (body * 2)) & (upper_shadow <= (candle_range * 0.1)) & (body <= (candle_range * 0.35))).astype(int)

    # 4. Inverted Hammer
    working_df['Pattern_Inverted_Hammer'] = ((upper_shadow >= (body * 2)) & (lower_shadow <= (candle_range * 0.1)) & (body <= (candle_range * 0.35))).astype(int)

    # 5. Hanging Man
    working_df['Pattern_Hanging_Man'] = (is_bearish & (lower_shadow >= (body * 2)) & (upper_shadow <= (candle_range * 0.1))).astype(int)

    # 6. Shooting Star
    working_df['Pattern_Shooting_Star'] = (is_bearish & (upper_shadow >= (body * 2)) & (lower_shadow <= (candle_range * 0.1))).astype(int)

    # 7. Spinning Top
    working_df['Pattern_Spinning_Top'] = ((body <= (candle_range * 0.3)) & (upper_shadow >= (body * 0.5)) & (lower_shadow >= (body * 0.5)) & (working_df['Pattern_Doji'] == 0)).astype(int)

    # Multi-bar shifts for sequential structural verification
    p_bull_eng = np.zeros(len(working_df), dtype=int)
    p_bear_eng = np.zeros(len(working_df), dtype=int)
    p_morning = np.zeros(len(working_df), dtype=int)
    p_evening = np.zeros(len(working_df), dtype=int)
    p_harami = np.zeros(len(working_df), dtype=int)
    p_piercing = np.zeros(len(working_df), dtype=int)
    p_dark_cloud = np.zeros(len(working_df), dtype=int)
    p_three_soldiers = np.zeros(len(working_df), dtype=int)
    p_three_crows = np.zeros(len(working_df), dtype=int)
    p_tweezer_top = np.zeros(len(working_df), dtype=int)
    p_tweezer_bottom = np.zeros(len(working_df), dtype=int)
    p_inside_up = np.zeros(len(working_df), dtype=int)
    p_inside_down = np.zeros(len(working_df), dtype=int)
    p_outside_up = np.zeros(len(working_df), dtype=int)
    p_outside_down = np.zeros(len(working_df), dtype=int)

    # Iterative loop execution safely optimized
    for i in range(2, len(working_df)):
        # 8. Bullish Engulfing
        if is_bearish[i-1] and is_bullish[i] and C[i] >= O[i-1] and O[i] <= C[i-1]:
            p_bull_eng[i] = 1

        # 9. Bearish Engulfing
        if is_bullish[i-1] and is_bearish[i] and C[i] <= O[i-1] and O[i] >= C[i-1]:
            p_bear_eng[i] = 1

        # 10. Harami
        if body[i-1] > avg_body[i-1] and body[i] < body[i-1] and body_max[i] <= body_max[i-1] and body_min[i] >= body_min[i-1]:
            p_harami[i] = 1

        # 11. Piercing Line
        if is_bearish[i-1] and is_bullish[i] and O[i] < L[i-1] and C[i] > (O[i-1] + C[i-1])/2 and C[i] < O[i-1]:
            p_piercing[i] = 1

        # 12. Dark Cloud Cover
        if is_bullish[i-1] and is_bearish[i] and O[i] > H[i-1] and C[i] < (O[i-1] + C[i-1])/2 and C[i] > O[i-1]:
            p_dark_cloud[i] = 1

        # 13. Morning Star
        if is_bearish[i-2] and body[i-1] < (avg_body[i-2] * 0.5) and is_bullish[i] and C[i] > (O[i-2] + C[i-2])/2:
            p_morning[i] = 1

        # 14. Evening Star
        if is_bullish[i-2] and body[i-1] < (avg_body[i-2] * 0.5) and is_bearish[i] and C[i] < (O[i-2] + C[i-2])/2:
            p_evening[i] = 1

        # 15. Three White Soldiers
        if is_bullish[i-2] and is_bullish[i-1] and is_bullish[i] and C[i] > C[i-1] and C[i-1] > C[i-2]:
            p_three_soldiers[i] = 1

        # 16. Three Black Crows
        if is_bearish[i-2] and is_bearish[i-1] and is_bearish[i] and C[i] < C[i-1] and C[i-1] < C[i-2]:
            p_three_crows[i] = 1

        # 17. Tweezer Top
        if np.abs(H[i] - H[i-1]) <= (candle_range[i] * 0.05) and is_bullish[i-1] and is_bearish[i]:
            p_tweezer_top[i] = 1

        # 18. Tweezer Bottom
        if np.abs(L[i] - L[i-1]) <= (candle_range[i] * 0.05) and is_bearish[i-1] and is_bullish[i]:
            p_tweezer_bottom[i] = 1

        # 19. Three Inside Up
        harami_up_check = (body[i-1] < body[i-2]) and (body_max[i-1] <= body_max[i-2])
        if is_bearish[i-2] and harami_up_check and is_bullish[i] and C[i] > C[i-1]:
            p_inside_up[i] = 1

        # 20. Three Inside Down
        harami_down_check = (body[i-1] < body[i-2]) and (body_min[i-1] >= body_min[i-2])
        if is_bullish[i-2] and harami_down_check and is_bearish[i] and C[i] < C[i-1]:
            p_inside_down[i] = 1

        # 21. Three Outside Up
        if is_bearish[i-2] and is_bullish[i-1] and C[i-1] >= O[i-2] and is_bullish[i] and C[i] > C[i-1]:
            p_outside_up[i] = 1

        # 22. Three Outside Down
        if is_bullish[i-2] and is_bearish[i-1] and C[i-1] <= O[i-2] and is_bearish[i] and C[i] < C[i-1]:
            p_outside_down[i] = 1

    # Assign mapped patterns safely to features space
    working_df['Pattern_Bullish_Engulfing'] = p_bull_eng
    working_df['Pattern_Bearish_Engulfing'] = p_bear_eng
    working_df['Pattern_Harami'] = p_harami
    working_df['Pattern_Piercing'] = p_piercing
    working_df['Pattern_Dark_Cloud'] = p_dark_cloud
    working_df['Pattern_Morning_Star'] = p_morning
    working_df['Pattern_Evening_Star'] = p_evening
    working_df['Pattern_Three_White_Soldiers'] = p_three_soldiers
    working_df['Pattern_Three_Black_Crows'] = p_three_crows
    working_df['Pattern_Tweezer_Top'] = p_tweezer_top
    working_df['Pattern_Tweezer_Bottom'] = p_tweezer_bottom
    working_df['Pattern_Three_Inside_Up'] = p_inside_up
    working_df['Pattern_Three_Inside_Down'] = p_inside_down
    working_df['Pattern_Three_Outside_Up'] = p_outside_up
    working_df['Pattern_Three_Outside_Down'] = p_outside_down

    pattern_cols = [col for col in working_df.columns if col.startswith('Pattern_')]
    total_detections = working_df[pattern_cols].sum().sum()

    print("\n==========================================================")
    print("📊 PHASE 3: CANDLESTICK INTELLIGENCE SIGNAL DETECTOR REPORT")
    print("==========================================================")
    print(f"  • Candlestick Vector Space Formed: {len(pattern_cols)} Patterns Active")
    print(f"  • Global Formations Flagged     : {total_detections} Occurrences")
    print("==========================================================")
    print("✅ Phase 3 candlestick feature generation complete without errors.")

    return working_df

# Trigger Phase 3 Execution safely
df_1h = execute_phase_3_candlestick_engine(df_1h)

🕯️ PHASE 3: INITIALIZING ARCHITECTURAL CANDLESTICK DETECTION

📊 PHASE 3: CANDLESTICK INTELLIGENCE SIGNAL DETECTOR REPORT
  • Candlestick Vector Space Formed: 22 Patterns Active
  • Global Formations Flagged     : 174471 Occurrences
✅ Phase 3 candlestick feature generation complete without errors.


In [6]:
# ==============================================================================
# 🥇 PHASE 4: TECHNICAL INDICATORS ENGINE (1H TIMEFRAME)
# Folder Mapping: src/indicators/indicator_engine.py & individual component files
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("⚙️ PHASE 4: COMPUTING STRUCTURAL TECHNICAL INDICATORS")
print("==========================================================")

def execute_phase_4_indicator_engine(df):
    """
    Vectorized structural indicator calculator implementing 9 specified trading
    metrics using native pandas and numpy window matrices.
    """
    working_df = df.copy()

    # Pre-extract values for fast matrix calculations
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()
    V = working_df['Volume'].to_numpy()

    # --------------------------------------------------------------------------
    # 1. Exponential Moving Averages (EMA20, EMA50, EMA200)
    # --------------------------------------------------------------------------
    print("   • Calculating Trend Lifelines: EMA20, EMA50, EMA200...")
    working_df['Indicator_EMA20'] = working_df['Close'].ewm(span=20, adjust=False).mean()
    working_df['Indicator_EMA50'] = working_df['Close'].ewm(span=50, adjust=False).mean()
    working_df['Indicator_EMA200'] = working_df['Close'].ewm(span=200, adjust=False).mean()

    # --------------------------------------------------------------------------
    # 2. Relative Strength Index (RSI - 14 Period Wilders Exponential Smoothing)
    # --------------------------------------------------------------------------
    print("   • Calculating Momentum Spaces: RSI14...")
    delta = working_df['Close'].diff().to_numpy()
    gain = np.where(delta > 0, delta, 0.0)
    loss = np.where(delta < 0, -delta, 0.0)

    avg_gain = pd.Series(gain).ewm(com=13, adjust=False).mean().to_numpy()
    avg_loss = pd.Series(loss).ewm(com=13, adjust=False).mean().to_numpy()
    # Avoid zero division inside oscillator limits
    avg_loss = np.where(avg_loss == 0, 1e-5, avg_loss)

    rs = avg_gain / avg_loss
    working_df['Indicator_RSI'] = 100 - (100 / (1 + rs))

    # --------------------------------------------------------------------------
    # 3. Moving Average Convergence Divergence (MACD 12, 26, 9)
    # --------------------------------------------------------------------------
    print("   • Calculating Volatility Waves: MACD Line, Signal Line, Hist...")
    ema12 = working_df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = working_df['Close'].ewm(span=26, adjust=False).mean()

    working_df['Indicator_MACD'] = ema12 - ema26
    working_df['Indicator_MACD_Signal'] = working_df['Indicator_MACD'].ewm(span=9, adjust=False).mean()
    working_df['Indicator_MACD_Hist'] = working_df['Indicator_MACD'] - working_df['Indicator_MACD_Signal']

    # --------------------------------------------------------------------------
    # 4. Average True Range (ATR - 14 Period True Range Bound)
    # --------------------------------------------------------------------------
    print("   • Calculating Risk Limits: ATR14...")
    prev_close = working_df['Close'].shift(1).to_numpy()
    prev_close[0] = C[0] # Handle boundary zero artifact

    tr1 = H - L
    tr2 = np.abs(H - prev_close)
    tr3 = np.abs(L - prev_close)

    true_range = np.maximum(tr1, np.maximum(tr2, tr3))
    working_df['Indicator_ATR'] = pd.Series(true_range).ewm(span=14, adjust=False).mean().to_numpy()

    # --------------------------------------------------------------------------
    # 5. Average Directional Index (ADX - 14 Period Strength Vector)
    # --------------------------------------------------------------------------
    print("   • Calculating Directional Forces: ADX14...")
    up_move = working_df['High'].diff().to_numpy()
    down_move = working_df['Low'].diff().to_numpy()
    # Mask negative variations
    up_move[0] = down_move[0] = 0.0

    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    atr_smooth = working_df['Indicator_ATR'].to_numpy()
    atr_smooth = np.where(atr_smooth == 0, 1e-5, atr_smooth) # Boundary safe rule

    plus_di = 100 * (pd.Series(plus_dm).ewm(span=14, adjust=False).mean().to_numpy() / atr_smooth)
    minus_di = 100 * (pd.Series(minus_dm).ewm(span=14, adjust=False).mean().to_numpy() / atr_smooth)

    di_sum = plus_di + minus_di
    di_sum = np.where(di_sum == 0, 1e-5, di_sum)
    dx = 100 * (np.abs(plus_di - minus_di) / di_sum)
    working_df['Indicator_ADX'] = pd.Series(dx).ewm(span=14, adjust=False).mean().to_numpy()

    # --------------------------------------------------------------------------
    # 6. Volume Weighted Average Price (VWAP - Cumulative Session Variant)
    # --------------------------------------------------------------------------
    print("   • Calculating Institutional Anchors: VWAP...")
    typical_price = (H + L + C) / 3.0
    cum_tp_v = (typical_price * V).cumsum()
    cum_v = V.cumsum()
    cum_v = np.where(cum_v == 0, 1e-5, cum_v)
    working_df['Indicator_VWAP'] = cum_tp_v / cum_v

    # --------------------------------------------------------------------------
    # 7. On-Balance Volume (OBV - Momentum Flow Strategy Matrix)
    # --------------------------------------------------------------------------
    print("   • Calculating Order Flows: OBV Accumulation...")
    obv = np.zeros(len(working_df))
    for i in range(1, len(working_df)):
        if C[i] > C[i-1]:
            obv[i] = obv[i-1] + V[i]
        elif C[i] < C[i-1]:
            obv[i] = obv[i-1] - V[i]
        else:
            obv[i] = obv[i-1]
    working_df['Indicator_OBV'] = obv

    # Handle nan fills across historical back-step boundaries
    indicator_cols = [col for col in working_df.columns if col.startswith('Indicator_')]
    working_df[indicator_cols] = working_df[indicator_cols].ffill().bfill().fillna(0.0)

    print("\n==========================================================")
    print("📊 PHASE 4: TECHNICAL INDICATORS EXTRACTION MATRIX REPORT")
    print("==========================================================")
    print(f"  • Mathematical Oscillators Locked: {len(indicator_cols)} Indicator Streams")
    print(f"  • Structural Array Completeness  : 100.00% Clean Numerical Space")
    print("==========================================================")
    print("✅ Phase 4 Technical Engineering layer closed successfully.")

    return working_df

# Trigger Phase 4 Execution safely
df_1h = execute_phase_4_indicator_engine(df_1h)

⚙️ PHASE 4: COMPUTING STRUCTURAL TECHNICAL INDICATORS
   • Calculating Trend Lifelines: EMA20, EMA50, EMA200...
   • Calculating Momentum Spaces: RSI14...
   • Calculating Volatility Waves: MACD Line, Signal Line, Hist...
   • Calculating Risk Limits: ATR14...
   • Calculating Directional Forces: ADX14...
   • Calculating Institutional Anchors: VWAP...
   • Calculating Order Flows: OBV Accumulation...

📊 PHASE 4: TECHNICAL INDICATORS EXTRACTION MATRIX REPORT
  • Mathematical Oscillators Locked: 11 Indicator Streams
  • Structural Array Completeness  : 100.00% Clean Numerical Space
✅ Phase 4 Technical Engineering layer closed successfully.


In [7]:
# ==============================================================================
# 🥇 PHASE 5: HISTORICAL STATISTICS ENGINE (1H TIMEFRAME)
# Folder Mapping: src/statistics/statistics_engine.py & structural files
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("📊 PHASE 5: EVALUATING HISTORICAL PATTERN INTELLIGENCE STATS")
print("==========================================================")

def execute_phase_5_statistics_engine(df):
    """
    Evaluates the historical efficacy of each detected pattern based on a forward
    4-candle window using a strict 1:2 Risk-Reward boundary condition.
    """
    working_df = df.copy()

    # 1. Isolate pattern columns and price tracking targets
    pattern_cols = [col for col in working_df.columns if col.startswith('Pattern_')]

    # Pre-extract data vectors to avoid inner loop Pandas overhead
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()
    n_records = len(working_df)

    # 2. Pre-calculate outcomes for every bar to evaluate signals efficiently
    # 1 = Bullish win, -1 = Bearish win, 0 = Timeout/Loss
    forward_bullish_win = np.zeros(n_records, dtype=int)
    forward_bearish_win = np.zeros(n_records, dtype=int)

    print("[1/2] Computing forward 4-candle structural boundaries...")
    for i in range(n_records - 4):
        entry = O[i + 1] # Market entry at next open price boundary

        # Bullish Boundaries (Using previous bar Low as SL)
        sl_bull = L[i]
        risk_bull = entry - sl_bull
        if risk_bull <= 0: risk_bull = 1e-5
        tp_bull = entry + (2.0 * risk_bull)

        # Bearish Boundaries (Using previous bar High as SL)
        sl_bear = H[i]
        risk_bear = sl_bear - entry
        if risk_bear <= 0: risk_bear = 1e-5
        tp_bear = entry - (2.0 * risk_bear)

        # Scan next 4 candles for resolution
        for step in range(1, 5):
            idx = i + step
            # Check Bullish path
            if L[idx] <= sl_bull:
                break # Stopped out
            if H[idx] >= tp_bull:
                forward_bullish_win[i] = 1
                break

        for step in range(1, 5):
            idx = i + step
            # Check Bearish path
            if H[idx] >= sl_bear:
                break # Stopped out
            if L[idx] <= tp_bear:
                forward_bearish_win[i] = 1
                break

    # 3. Compile Performance Metrics Dictionary for XAI Support Layers
    print("[2/2] Generating comprehensive profile dictionary for active schemas...")
    stats_profile = {}

    for col in pattern_cols:
        indices = working_df[working_df[col] == 1].index.to_numpy()
        occurrences = len(indices)

        if occurrences == 0:
            stats_profile[col] = {"Occurrence": 0, "Success_Rate": 0.0, "Expected_Value": 0.0, "Reliability": "Low"}
            continue

        # Determine performance profile based on standard naming structures
        is_bull_pattern = any(x in col.lower() for x in ['bullish', 'hammer', 'morning', 'soldiers', 'inside_up', 'outside_up', 'tweezer_bottom'])

        wins = 0
        for idx in indices:
            if is_bull_pattern and forward_bullish_win[idx] == 1:
                wins += 1
            elif not is_bull_pattern and forward_bearish_win[idx] == 1:
                wins += 1

        success_rate = (wins / occurrences) * 100.0

        # Expected value calculation framework: (Win% * 2.0 RR) - (Loss% * 1.0 Risk)
        ev = ((success_rate / 100.0) * 2.0) - ((1.0 - (success_rate / 100.0)) * 1.0)

        reliability = "High" if success_rate >= 55.0 and occurrences >= 30 else "Moderate" if success_rate >= 45.0 else "Low"

        stats_profile[col] = {
            "Occurrence": occurrences,
            "Success_Rate": round(success_rate, 2),
            "Expected_Value": round(ev, 3),
            "Reliability": reliability
        }

    # Save dictionary metadata back to global script context for the downstream XAI layer
    working_df.attrs['pattern_stats_profile'] = stats_profile

    # Quick printout summary of top performing patterns detected
    print("\n==========================================================")
    print("🎯 PHASE 5: HISTORICAL INTELLIGENCE STATISTICAL HIGHLIGHTS")
    print("==========================================================")
    sorted_patterns = sorted(stats_profile.items(), key=lambda x: x[1]['Success_Rate'], reverse=True)[:3]
    for name, metrics in sorted_patterns:
        print(f"  • {name:<30} | Occur: {metrics['Occurrence']:<4} | WinRate: {metrics['Success_Rate']}% | EV: {metrics['Expected_Value']}")
    print("==========================================================")
    print("✅ Phase 5 historical statistics locked into dataframe properties.")

    return working_df

# Trigger Phase 5 Execution safely
df_1h = execute_phase_5_statistics_engine(df_1h)

📊 PHASE 5: EVALUATING HISTORICAL PATTERN INTELLIGENCE STATS
[1/2] Computing forward 4-candle structural boundaries...
[2/2] Generating comprehensive profile dictionary for active schemas...

🎯 PHASE 5: HISTORICAL INTELLIGENCE STATISTICAL HIGHLIGHTS
  • Pattern_Spinning_Top           | Occur: 21509 | WinRate: 25.72% | EV: -0.228
  • Pattern_Doji                   | Occur: 14261 | WinRate: 22.99% | EV: -0.31
  • Pattern_Harami                 | Occur: 12471 | WinRate: 22.91% | EV: -0.313
✅ Phase 5 historical statistics locked into dataframe properties.


In [8]:
# ==============================================================================
# 🥇 PHASE 6 UPGRADE: MULTI-TASK FEATURE SELECTION & TARGET LABELLING
# Folder Mapping: src/features/feature_engineering.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🔧 PHASE 6: GENERATING MULTI-TASK TARGET LABELS")
print("==========================================================")

def execute_phase_6_multi_task(df):
    working_df = df.copy()

    # 1. Base transformations
    working_df['Feature_Log_Volume'] = np.log1p(np.maximum(working_df['Volume'].to_numpy(), 0.0))
    obv_array = working_df['Indicator_OBV'].to_numpy()
    working_df['Feature_Log_OBV'] = np.sign(obv_array) * np.log1p(np.abs(obv_array))

    # 2. Continuous Metric Scaling Baseline
    continuous_features = [
        'Open', 'High', 'Low', 'Close',
        'Indicator_EMA20', 'Indicator_EMA50', 'Indicator_EMA200',
        'Indicator_RSI', 'Indicator_MACD', 'Indicator_MACD_Signal', 'Indicator_MACD_Hist',
        'Indicator_ATR', 'Indicator_ADX', 'Indicator_VWAP', 'Feature_Log_Volume', 'Feature_Log_OBV'
    ]
    for col in continuous_features:
        col_mean = working_df[col].mean()
        col_std = working_df[col].std()
        if col_std == 0: col_std = 1e-5
        working_df[f'Scaled_{col}'] = (working_df[col] - col_mean) / col_std

    # --------------------------------------------------------------------------
    # DUAL TARGET LOGIC FOR CLASSIFICATION & REGRESSION
    # --------------------------------------------------------------------------
    next_close = working_df['Close'].shift(-1).to_numpy()
    current_close = working_df['Close'].to_numpy()

    # A. Classification Target: Directional Binary Vector (Up=1, Down=0)
    working_df['Target_Class'] = (next_close > current_close).astype(int)

    # B. Regression Target: Continuous Magnitude Return Vector (Price Difference)
    working_df['Target_Reg'] = next_close - current_close

    # Clean boundary constraints
    working_df = working_df.iloc[:-1].reset_index(drop=True)
    final_features = [col for col in working_df.columns if col.startswith('Scaled_') or col.startswith('Pattern_')]

    print("✅ Dual Targets Created: 'Target_Class' & 'Target_Reg' successfully mapped.")
    return working_df, final_features

df_1h, feature_columns_list = execute_phase_6_multi_task(df_1h)

🔧 PHASE 6: GENERATING MULTI-TASK TARGET LABELS
✅ Dual Targets Created: 'Target_Class' & 'Target_Reg' successfully mapped.


In [9]:
# ==============================================================================
# 🥇 PHASE 7: PURE STANDALONE 2D MATRIX SPLITTER (XGBOOST FLATTENED)
# Folder Mapping: src/data/sequence_generator.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("⚙️ PHASE 7: BUILDING CLEAN FLATTENED DATA SPLITS FOR XGBOOST")
print("==========================================================")

# 1. Chronological Splitting Boundaries (70% Train | 15% Val | 15% Test)
total_samples = len(df_1h)
train_end = int(total_samples * 0.70)
val_end = int(total_samples * 0.85)

# 2. Extract Flat 2D Matrices Directly from Feature List
# XGBoost requires standard 2D matrices [Samples, Features]
X_flat_all = df_1h[feature_columns_list].to_numpy(dtype=np.float32)
y_flat_all = df_1h['Target_Class'].to_numpy(dtype=np.int32)

# 3. Create Clean Slices Without Window Leaks
X_train = X_flat_all[:train_end]
y_train = y_flat_all[:train_end]

X_val = X_flat_all[train_end:val_end]
y_val = y_flat_all[train_end:val_end]

X_test = X_flat_all[val_end:]
y_test = y_flat_all[val_end:]

print("\n📊 XGBOOST FLATTENED DATA DISTRIBUTION SUMMARY:")
print("==========================================================")
print(f"  • X_train Shape : {X_train.shape} | y_train Shape : {y_train.shape}")
print(f"  • X_val Shape   : {X_val.shape}  | y_val Shape   : {y_val.shape}")
print(f"  • X_test Shape  : {X_test.shape}  | y_test Shape  : {y_test.shape}")
print("==========================================================")
print("✅ Phase 7 permanently updated. 2D Data structures ready for XGBoost Model.")

⚙️ PHASE 7: BUILDING CLEAN FLATTENED DATA SPLITS FOR XGBOOST

📊 XGBOOST FLATTENED DATA DISTRIBUTION SUMMARY:
  • X_train Shape : (87643, 38) | y_train Shape : (87643,)
  • X_val Shape   : (18781, 38)  | y_val Shape   : (18781,)
  • X_test Shape  : (18781, 38)  | y_test Shape  : (18781,)
✅ Phase 7 permanently updated. 2D Data structures ready for XGBoost Model.


In [10]:
# ==============================================================================
# 🥇 PHASE 8: XGBOOST HYPERPARAMETER & ARCHITECTURE SPECIFICATION
# Folder Mapping: src/models/xgboost_config.py
# ==============================================================================
import xgboost as xgb

print("==========================================================")
print("🧠 PHASE 8: CONFIGURING GRADIENT BOOSTED TREE PARAMETERS")
print("==========================================================")

def configure_xgboost_engine():
    """
    Defines hyperparameter maps optimized for high-dimensional tabular
    financial indicators, bypassing complex neural network layers.
    """
    # 1. Defining production-grade hyperparameters for financial arrays
    xgb_params = {
        'n_estimators': 300,        # Number of boosting rounds (trees)
        'max_depth': 6,             # Prevents tree over-fitting on financial noise
        'learning_rate': 0.03,      # Step size shrinkage to secure local minima
        'subsample': 0.8,           # Row subsampling to add variance protection
        'colsample_bytree': 0.8,    # Feature subsampling per tree (prevents dominance)
        'tree_method': 'hist',      # High-performance histogram optimized processing
        'random_state': 42,
        'n_jobs': -1                # Utilizes all CPU cores for parallel processing
    }

    print("\n📊 PHASE 8: XGBOOST PARAMS CONFIGURATION SCHEMATIC")
    print("==========================================================")
    for key, value in xgb_params.items():
        print(f"  • {key.ljust(18)} : {value}")
    print("==========================================================")
    print("✅ Phase 8 config completed. Parameters locked for Multi-Task execution.")

    return xgb_params

# Instantiate the parameter map
xgb_config_parameters = configure_xgboost_engine()

🧠 PHASE 8: CONFIGURING GRADIENT BOOSTED TREE PARAMETERS

📊 PHASE 8: XGBOOST PARAMS CONFIGURATION SCHEMATIC
  • n_estimators       : 300
  • max_depth          : 6
  • learning_rate      : 0.03
  • subsample          : 0.8
  • colsample_bytree   : 0.8
  • tree_method        : hist
  • random_state       : 42
  • n_jobs             : -1
✅ Phase 8 config completed. Parameters locked for Multi-Task execution.


In [11]:
# ==============================================================================
# 🥇 PHASE 9: ADVANCED MULTI-TASK XGBOOST WITH STRUCTURAL 1:2 R:R CRITERIA
# Folder Mapping: src/models/xgboost_trainer.py
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("==========================================================")
print("🧠 PHASE 9: EMBEDDING STRUCTURAL 1:2 R:R MATRIX INTO XGBOOST")
print("==========================================================")

# ------------------------------------------------------------------------------
# Step 1: Pre-compute Continuous Targets & Metadata Fields
# ------------------------------------------------------------------------------
# Target_Class is direction (0 or 1). For regression magnitude, we use next candle returns.
if 'Return' in df_1h.columns:
    df_1h['Target_Reg'] = df_1h['Return'].shift(-1).fillna(0).astype(np.float32)
else:
    df_1h['Target_Reg'] = df_1h['Close'].pct_change().shift(-1).fillna(0).astype(np.float32)

# ------------------------------------------------------------------------------
# Step 2: Extract Matrices & Apply Z-Score Scaling
# ------------------------------------------------------------------------------
X_raw_matrix = df_1h[feature_columns_list].to_numpy(dtype=np.float32)
y_cls_vector = df_1h['Target_Class'].to_numpy(dtype=np.int32)
y_reg_vector = df_1h['Target_Reg'].to_numpy(dtype=np.float32)

# Fit and apply scaler parameters using the training dimensions from Phase 7
scaler = StandardScaler()
X_scaled_all = scaler.fit_transform(X_raw_matrix)

# Slice chronological arrays safely to feed the trainer
X_train_flat, y_train_cls, y_train_reg = X_scaled_all[:train_end], y_cls_vector[:train_end], y_reg_vector[:train_end]
X_val_flat, y_val_cls, y_val_reg = X_scaled_all[train_end:val_end], y_cls_vector[train_end:val_end], y_reg_vector[train_end:val_end]

# ------------------------------------------------------------------------------
# Step 3: Implement 1:2 Risk-to-Reward Custom Sample Weighting Matrix
# ------------------------------------------------------------------------------
print("[INFO] Enforcing 1:2 R:R constraints via dynamic sample-weight matrices...")

# Logic: Agar actual movement ka variance edge strong hai (1:2 criteria meet ho raha hai),
# toh un bars ka sample weight scale up (2.5x penalty structural protection) kar do.
# Yeh directly custom loss structure ki tarah mathematically behavior optimize karega.
sample_weights_train = np.ones(len(y_train_reg), dtype=np.float32)

# Identify samples where expected target outcome yields deep return variance
mean_absolute_target = np.mean(np.abs(y_train_reg))
heavy_reward_indices = np.where(np.abs(y_train_reg) >= (2.0 * mean_absolute_target))[0]

# Inject the 2.5x penalty weight to force decision trees to learn these high R:R zones
sample_weights_train[heavy_reward_indices] = 2.5

print(f"✓ R:R Strategy Matrix Scaled! {len(heavy_reward_indices)} high-yield bars up-weighted to 2.5x.")
print("==========================================================")
print("✅ Phase 9 matrix conversion successful. 2D arrays and weights ready for training.")

🧠 PHASE 9: EMBEDDING STRUCTURAL 1:2 R:R MATRIX INTO XGBOOST
[INFO] Enforcing 1:2 R:R constraints via dynamic sample-weight matrices...
✓ R:R Strategy Matrix Scaled! 10959 high-yield bars up-weighted to 2.5x.
✅ Phase 9 matrix conversion successful. 2D arrays and weights ready for training.


In [19]:
# RSI aur EMA indicators data frame me properly saved hone chahiye
feature_columns_list = ['Body_to_Range', 'Upper_Wick_Ratio', 'Lower_Wick_Ratio', 'Log_Returns', 'Volume_MOM', 'RSI', 'EMA_Trend_Signal']

In [24]:
# ==============================================================================
# 🥇 PHASE 10: AUTO-FEATURE REBUILD & TRAINING ENGINE (CRASH PROOF)
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import RobustScaler

print("==========================================================")
print("⚙️ PHASE 10: REBUILDING FEATURES & TRAINING CLASSIFIER")
print("==========================================================")

# 1. Load Raw Data safely from Drive or Memory
if 'df_1h' in locals() or 'df_1h' in globals():
    active_df = df_1h.copy()
else:
    data_path = '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv'
    if os.path.exists(data_path):
        active_df = pd.read_csv(data_path)
        print("✓ Raw CSV loaded directly from Google Drive.")
    else:
        raise FileNotFoundError("❌ Dataframe 'df_1h' not found. Please run your data loading cells first!")

# 2. 🔥 FORCE REBUILDING THE 5 CORE QUANT FEATURES (Fixes the KeyError)
print("⚙️ Recalculating missing feature pillars dynamically...")
active_df = active_df.astype({'Open': 'float64', 'High': 'float64', 'Low': 'float64', 'Close': 'float64', 'Volume': 'float64'})

active_df['Body'] = (active_df['Close'] - active_df['Open']).abs()
active_df['Total_Range'] = active_df['High'] - active_df['Low']
active_df['Total_Range'] = active_df['Total_Range'].replace(0, 1e-9)

active_df['Body_to_Range'] = active_df['Body'] / active_df['Total_Range']
active_df['Upper_Wick'] = active_df['High'] - active_df[['Open', 'Close']].max(axis=1)
active_df['Lower_Wick'] = active_df[['Open', 'Close']].min(axis=1) - active_df['Low']

active_df['Upper_Wick_Ratio'] = active_df['Upper_Wick'] / active_df['Total_Range']
active_df['Lower_Wick_Ratio'] = active_df['Lower_Wick'] / active_df['Total_Range']
active_df['Log_Returns'] = np.log(active_df['Close'] / active_df['Close'].shift(1))
active_df['Volume_MOM'] = active_df['Volume'] / active_df['Volume'].shift(1).replace(0, 1e-9)

# Create Binary Target
active_df['Target_Cls'] = (active_df['Close'].shift(-1) > active_df['Close']).astype(int)
active_df.dropna(inplace=True)

# 3. Features Lock
feature_columns_list = ['Body_to_Range', 'Upper_Wick_Ratio', 'Lower_Wick_Ratio', 'Log_Returns', 'Volume_MOM']
print(f"✓ Feature pillars rebuilt. Total active columns: {len(feature_columns_list)}")

# 4. Train/Test Chronological Split (80/20)
train_size = int(len(active_df) * 0.80)
train_df = active_df.iloc[:train_size].copy()
test_df = active_df.iloc[train_size:].copy()

X_train_raw = train_df[feature_columns_list].values
y_train_clean = train_df['Target_Cls'].values
X_test_raw = test_df[feature_columns_list].values
y_test_clean = test_df['Target_Cls'].values

# 5. Robust Scaling
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 6. Initialize Tuned Classifier (Robust to Noise)
xgb_classifier = XGBClassifier(
    n_estimators=100,      # Overfitting control
    max_depth=3,           # Less depth to avoid noise
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    eval_metric='logloss'
)

# 7. Fit the Model
print("   • Training Tuned XGBoost Classifier...")
xgb_classifier.fit(X_train_scaled, y_train_clean)

# 8. Export in Friend's exact format
os.makedirs('models', exist_ok=True)
model_filename = "models/gold_1h_model.pkl"
with open(model_filename, 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'model': xgb_classifier,
        'features': feature_columns_list
    }, f)

print("==========================================================")
print(f"✅ Rebuild complete! Model successfully exported: {model_filename}")
print("==========================================================")

⚙️ PHASE 10: REBUILDING FEATURES & TRAINING CLASSIFIER
⚙️ Recalculating missing feature pillars dynamically...
✓ Feature pillars rebuilt. Total active columns: 5
   • Training Tuned XGBoost Classifier...
✅ Rebuild complete! Model successfully exported: models/gold_1h_model.pkl


In [28]:
# ==============================================================================
# 🥇 PHASE 11: SINGLE-TASK EVALUATION ENGINE (CLASSIFICATION ONLY)
# ==============================================================================
import pickle
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("   • Running inference steps for Classification Layer...")

# Load the model, scaler, and feature list from the saved pickle file (Phase 10)
model_filename = "models/gold_1h_model.pkl"
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}. Please run Phase 10 first to train and save the model.")

with open(model_filename, 'rb') as f:
    saved_model_data = pickle.load(f)

# Extract the trained classifier, scaler, and feature list
xgb_classifier = saved_model_data['model']
scaler = saved_model_data['scaler']
feature_columns_list_from_model = saved_model_data['features']

# Re-create the test_df based on the same logic as Phase 10
# (assuming df_1h is available globally from earlier phases)
active_df = df_1h.copy()

# --- Replicate Feature Recalculation from Phase 10 START ---
# Ensure numerical features are float64 for calculations
active_df = active_df.astype({'Open': 'float64', 'High': 'float64', 'Low': 'float64', 'Close': 'float64', 'Volume': 'float64'})

active_df['Body'] = (active_df['Close'] - active_df['Open']).abs()
active_df['Total_Range'] = active_df['High'] - active_df['Low']
active_df['Total_Range'] = active_df['Total_Range'].replace(0, 1e-9)

active_df['Body_to_Range'] = active_df['Body'] / active_df['Total_Range']
active_df['Upper_Wick'] = active_df['High'] - active_df[['Open', 'Close']].max(axis=1)
active_df['Lower_Wick'] = active_df[['Open', 'Close']].min(axis=1) - active_df['Low']

active_df['Upper_Wick_Ratio'] = active_df['Upper_Wick'] / active_df['Total_Range']
active_df['Lower_Wick_Ratio'] = active_df['Lower_Wick'] / active_df['Total_Range']
active_df['Log_Returns'] = np.log(active_df['Close'] / active_df['Close'].shift(1))
active_df['Volume_MOM'] = active_df['Volume'] / active_df['Volume'].shift(1).replace(0, 1e-9)
# --- Replicate Feature Recalculation from Phase 10 END ---

active_df['Target_Cls'] = (active_df['Close'].shift(-1) > active_df['Close']).astype(int)
active_df.dropna(inplace=True)

train_size = int(len(active_df) * 0.80)
test_df = active_df.iloc[train_size:].copy()

# Prepare X_test_scaled using the loaded scaler and feature list
X_test_raw_for_eval = test_df[feature_columns_list_from_model].values
X_test_scaled = scaler.transform(X_test_raw_for_eval)
y_test_clean = test_df['Target_Cls'].values

# Generate Predictions
test_probs = xgb_classifier.predict_proba(X_test_scaled)[:, 1]
test_preds = (test_probs >= 0.50).astype(int)

# Calculate Metrics
test_acc = accuracy_score(y_test_clean, test_preds)
test_prec = precision_score(y_test_clean, test_preds, zero_division=0)
test_rec = recall_score(y_test_clean, test_preds, zero_division=0)
test_f1 = f1_score(y_test_clean, test_preds, zero_division=0)

print("\n==========================================================")
print("🎯 PHASE 11: XGBOOST CLASSIFICATION PERFORMANCE REPORT")
print("==========================================================")
print(f"  • Test Accuracy          : {test_acc * 100.0:.2f}%")
print(f"  • Test Precision Score   : {test_prec:.4f}")
print(f"  • Test Recall (Sens.)    : {test_rec:.4f}")
print(f"  • Test F1-Score          : {test_f1:.4f}")
print("==========================================================")

   • Running inference steps for Classification Layer...

🎯 PHASE 11: XGBOOST CLASSIFICATION PERFORMANCE REPORT
  • Test Accuracy          : 51.26%
  • Test Precision Score   : 0.5227
  • Test Recall (Sens.)    : 0.4931
  • Test F1-Score          : 0.5075


In [30]:
import numpy as np
import pandas as pd

print("==========================================================")
print("🥇 PHASE 12: VECTORIZED BACKTESTING LAYER (BINARY SIGNALS ONLY)")
print("==========================================================")

# The original 'backtest_df' slicing using 'val_end' from Phase 7
# resulted in a dataframe with a different number of rows than 'test_probs'
# generated in Phase 11. To align them, we directly use the 'test_df'
# that was constructed in Phase 11 and corresponds to 'test_probs'.
# test_df is already available in the kernel from Phase 11.
backtest_df = test_df.copy().reset_index(drop=True)

# Map dynamic trading signals (1 for Buy, -1 for Short)
backtest_df['Model_Signal_Prob'] = test_probs
backtest_df['Trading_Signal'] = np.where(backtest_df['Model_Signal_Prob'] >= 0.50, 1, -1)

# Returns computation
backtest_df['Market_Returns'] = backtest_df['Close'].pct_change().shift(-1).fillna(0.0)
backtest_df['Strategy_Returns'] = backtest_df['Trading_Signal'] * backtest_df['Market_Returns']

# Compounding Equity Curve
backtest_df['Cum_Market_Returns'] = (1.0 + backtest_df['Market_Returns']).cumprod() - 1.0
backtest_df['Cum_Strategy_Returns'] = (1.0 + backtest_df['Strategy_Returns']).cumprod() - 1.0

print("\n==========================================================")
print("📊 PHASE 12: BINARY ALGORITHMIC PERFORMANCE REPORT")
print("==========================================================")
print(f"  • Cumulative Benchmark Return : {backtest_df['Cum_Market_Returns'].iloc[-1]*100.0:.2f}%")
print(f"  • Cumulative Strategy Return  : {backtest_df['Cum_Strategy_Returns'].iloc[-1]*100.0:.2f}%")
print("==========================================================")

🥇 PHASE 12: VECTORIZED BACKTESTING LAYER (BINARY SIGNALS ONLY)

📊 PHASE 12: BINARY ALGORITHMIC PERFORMANCE REPORT
  • Cumulative Benchmark Return : 176.18%
  • Cumulative Strategy Return  : 34.01%


In [ ]:
# import os
# import zipfile
# from google.colab import files

# print("==========================================================")
# print("📦 CRASH-PROOF BUNDLE ZIP ENGINE (SHORTCUTS BYPASSED)")
# print("==========================================================")

# zip_filename = "Gold_AI_Trading_System_Complete.zip"

# # Target paths
# drive_folder = "/content/drive/MyDrive/Gold-AI-Trading-System"
# local_extensions = ('.pkl', '.csv', '.html', '.zip')
# ignored_extensions = ('.gslides', '.gdoc', '.gsheet') # 🚨 Google Drive shortcuts to bypass

# with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

#     # 1. Google Drive files packing with shortcut protection
#     if os.path.exists(drive_folder):
#         print("📁 Scanning Google Drive directory...")
#         for root, dirs, filenames in os.walk(drive_folder):
#             for file in filenames:
#                 if file.endswith(ignored_extensions):
#                     # Link shortcuts ko skip karo taaki OSError na aaye
#                     continue

#                 file_path = os.path.join(root, file)
#                 arcname = os.path.relpath(file_path, os.path.dirname(drive_folder))

#                 try:
#                     zipf.write(file_path, arcname)
#                     print(f"  • Added from Drive: {arcname}")
#                 except Exception as e:
#                     print(f"  ⚠️ Skipped file due to permissions: {file}")
#     else:
#         print("⚠️ Note: Google Drive folder path not found.")

#     # 2. Local workspace files packing
#     print("\n📁 Scanning local workspace for artifacts...")
#     for item in os.listdir():
#         if os.path.isfile(item) and item.endswith(local_extensions) and item != zip_filename:
#             zipf.write(item, os.path.join("Local_Workspace", item))
#             print(f"  • Added from Local: Local_Workspace/{item}")

# print("----------------------------------------------------------")
# print("🚀 Packing fully complete! Triggering automatic download...")
# files.download(zip_filename)